# Fair baselines, calibration, and locked-OOT validation

**Research question:** Does the nonlinear challenger add value over a credible binned-logit baseline in future cohorts?

This notebook is the review interface. Reusable transformations remain in `src/`, `scripts/`, and `sql/`; the notebook reads their committed evidence rather than creating a second implementation. Raw source data is intentionally not embedded.

## Hypotheses declared before interpretation

- A raw linear model understates a fair scorecard-like baseline.
- Ranking and probability calibration must be assessed separately.
- Policy cutoffs must be selected before locked OOT evaluation.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print(f'Repository: {ROOT.name}')


Repository: credit-risk-decision-lab


## Feature provenance

Every row states its raw source, transformation, decision-time cutoff, fit scope, missing-value treatment, and known leakage or proxy risk.

In [2]:
features = pd.read_csv(ROOT / 'reports' / 'feature_provenance.csv')
display(features)
display(features.groupby(['feature_family', 'model_role']).size().rename('features').reset_index())


,feature_name,feature_family,raw_source,grain,transformation,observation_cutoff,available_at_decision,fit_partition,missing_treatment,model_role,exclusion_reason,leakage_risk,proxy_or_fairness_risk,validation
0,numeric_static,application static,train_static_0_*.parquet,application,Numeric fields eligible under frozen manifest,at date_decision,yes,development,native missing for XGBoost; median/binning for...,model,NaN,medium: definitions reviewed for post-decision...,credit-history variables may create access pro...,"feature inventory, time stability, baseline co..."
1,categorical_static,application static,train_static_0_* and train_static_cb_0,application,Low-cardinality categories eligible under froz...,at date_decision,yes,development,unknown/encoded from development,model,NaN,medium,category values may proxy protected characteri...,cardinality and missingness gates
2,relative_date_D,date recency,source columns ending D,application,date_decision - source date in days,at date_decision,yes,development,native missing,model,NaN,low after relative conversion,history length can proxy age/tenure,post-decision date check
3,missing_indicator,coverage,selected source fields,application,1 when source value is missing,at date_decision,yes,development,explicit binary,linear baseline,NaN,medium: provider/product eligibility signal,coverage can proxy channel or geography,cohort missingness monitoring
4,quantile_bin,nonlinear baseline,eligible numeric features,application,Ten development-fitted quantile bins with one-...,at date_decision,yes,development,imputer inside pipeline,binned-logit baseline,NaN,low,inherits source feature risks,locked OOT baseline comparison
5,case_id,identifier,train_base.case_id,application,No transformation,at decision,no model use,none,not applicable,excluded,identifier,high: identifier leakage,identity,excluded by contract
6,WEEK_NUM/MONTH/date_decision,partition/time,train_base,application,Used only for chronological partitions,at decision,audit only,none,not applicable,excluded,partition leakage,high: direct cohort leakage,time may proxy policy changes,excluded by contract
7,sensitive-token fields,protected/proxy,source fields matching birth/marital/education...,application,No transformation,at decision,excluded,development,not applicable,excluded,sensitive/proxy governance,high governance risk,protected or close proxy,frozen exclusion list


,feature_family,model_role,features
0,application static,model,2
1,coverage,linear baseline,1
2,date recency,model,1
3,identifier,excluded,1
4,nonlinear baseline,binned-logit baseline,1
5,partition/time,excluded,1
6,protected/proxy,excluded,1


## Decision-time audit

A feature is eligible only when its information exists at the stated observation cutoff. Model-fitted bins, encoders, imputers, and thresholds use development data only. Outcomes and evaluation groups never enter the predictor matrix.

In [3]:
audit = features[['feature_name', 'observation_cutoff', 'available_at_decision', 'fit_partition', 'leakage_risk', 'validation']]
display(audit)


,feature_name,observation_cutoff,available_at_decision,fit_partition,leakage_risk,validation
0,numeric_static,at date_decision,yes,development,medium: definitions reviewed for post-decision...,"feature inventory, time stability, baseline co..."
1,categorical_static,at date_decision,yes,development,medium,cardinality and missingness gates
2,relative_date_D,at date_decision,yes,development,low after relative conversion,post-decision date check
3,missing_indicator,at date_decision,yes,development,medium: provider/product eligibility signal,cohort missingness monitoring
4,quantile_bin,at date_decision,yes,development,low,locked OOT baseline comparison
5,case_id,at decision,no model use,none,high: identifier leakage,excluded by contract
6,WEEK_NUM/MONTH/date_decision,at decision,audit only,none,high: direct cohort leakage,excluded by contract
7,sensitive-token fields,at decision,excluded,development,high governance risk,frozen exclusion list


## Committed analytical evidence

In [4]:
evidence_files = ["reports/binned_baseline_metrics.csv", "reports/model_metrics.csv"]
for relative in evidence_files:
    path = ROOT / relative
    display(Markdown(f'### `{relative}`'))
    if not path.exists():
        display(Markdown('*Not committed; reproduce locally using the README pipeline.*'))
    elif path.suffix == '.csv':
        display(pd.read_csv(path).head(20))
    elif path.suffix == '.json':
        payload = json.loads(path.read_text())
        display(payload)


### `reports/binned_baseline_metrics.csv`

,model,partition,rows,roc_auc,gini,average_precision,brier_score,ece_10,observed_default_rate,mean_predicted_pd,stability_score,stability_mean_gini,stability_slope,stability_falling_rate,stability_residual_std,stability_eligible_weeks
0,binned_logistic,calibration,171375,0.748312,0.496625,0.135931,0.041157,0.003243,0.045135,0.045135,0.468783,0.499113,0.001089,0.0,0.060659,14
1,binned_logistic,policy,73807,0.781532,0.563065,0.098284,0.022142,0.007331,0.023494,0.030824,0.510336,0.545917,0.005780,0.0,0.071163,9
2,binned_logistic,oot,151707,0.786973,0.573946,0.096438,0.020059,0.009564,0.021192,0.030756,0.544616,0.564582,0.003758,0.0,0.039932,14


### `reports/model_metrics.csv`

,model,partition,rows,roc_auc,gini,average_precision,brier_score,ece_10,observed_default_rate,mean_predicted_pd,stability_score,stability_mean_gini,stability_slope,stability_falling_rate,stability_residual_std,stability_eligible_weeks
0,linear,calibration,171375,0.684638,0.369277,0.102743,0.042102,0.006786,0.045135,0.045133,-0.553926,0.351747,-0.010088,-0.010088,0.035824,14
1,challenger,calibration,171375,0.792341,0.584682,0.178937,0.039965,0.002264,0.045135,0.045151,0.442820,0.581603,-0.001306,-0.001306,0.047654,14
2,linear,policy,73807,0.701059,0.402119,0.070026,0.022841,0.017845,0.023494,0.041338,0.342803,0.379440,0.021087,0.000000,0.073273,9
3,challenger,policy,73807,0.806500,0.613001,0.127346,0.022007,0.012297,0.023494,0.035791,0.576586,0.603925,0.000074,0.000000,0.054678,9
4,linear,oot,151707,0.711330,0.422660,0.064408,0.020801,0.020063,0.021192,0.041255,0.394292,0.418258,0.001415,0.000000,0.047932,14
5,challenger,oot,151707,0.825343,0.650685,0.138444,0.019726,0.012863,0.021192,0.034055,0.616423,0.639835,0.003515,0.000000,0.046825,14


## Decision

**XGBoost improves OOT ranking over binned logit, but calibration, maturity, fairness, and governance require shadow mode.**

### Claim boundary

The notebook distinguishes observed evidence from model output, simulated policy, and causal impact. A strong offline metric is not production exposure.